# MedTrack_DV — 03. Data Normalization (Building the Four Final Tables)

Builds the four analytical tables the project requires, following the doc's architecture:

```
CORE HMIS DATA
     |
 PATIENT / DEPARTMENT / RESOURCE
     |
  ADMISSION (spine)
     |
 READMISSION DATA / DISCHARGE DATA (bridged in, not merged blindly)
     |
FINAL DATA MODEL: Hospital Overview | Patient Flow | Department Analytics | Resource Utilization
```

**Grain of each table** (must hold exactly, verified with asserts below):
- Hospital Overview: 1 row = 1 admission
- Patient Flow: 1 row = 1 movement event
- Department Analytics: 1 row = 1 department + 1 day
- Resource Utilization: 1 row = 1 department + 1 day + 1 resource type

This notebook does NOT blindly merge the 3 datasets. HMIS builds all four tables on its own first;
Beds Management and the Readmission dataset are bridged in afterward, only where a defensible
mapping exists, and only as additive columns on top of the HMIS-built rows.

In [1]:
import pandas as pd
import re
import calendar
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
HMIS_PROCESSED_DIR        = PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics"
BEDS_PROCESSED_DIR        = PROCESSED_DIR / "Hospital Beds Management"
READMISSION_PROCESSED_DIR = PROCESSED_DIR / "Hospital Data for Patient Readmission Prediction"

In [2]:
# =========================================================
# LOAD CLEANED HMIS TABLES
# (dates get saved as text in CSV, so we re-parse them here)
# =========================================================
cleaned_dataframes = {}
for name in ['department','patient','employee','disease','insurance_provider','drug_manufacturer',
             'doctor','ward','drug','patient_insurance','bed','drug_inventory','staff_assignment',
             'admission','diagnostic_test','patient_diagnostic','prescription','billing','billing_detail']:
    cleaned_dataframes[name] = pd.read_csv(HMIS_PROCESSED_DIR / f"{name}.csv")

date_cols_by_table = {
    'patient': ['date_of_birth'], 'employee': ['date_of_joining'],
    'admission': ['admission_date', 'discharge_date'],
    'patient_insurance': ['policy_start_date', 'policy_end_date'],
    'drug_inventory': ['last_restock_date'], 'patient_diagnostic': ['test_date'],
    'billing': ['bill_date'],
}
for table, cols in date_cols_by_table.items():
    for c in cols:
        cleaned_dataframes[table][c] = pd.to_datetime(cleaned_dataframes[table][c])

admission = cleaned_dataframes['admission']
patient = cleaned_dataframes['patient']
department = cleaned_dataframes['department']
ward = cleaned_dataframes['ward']
bed = cleaned_dataframes['bed']
disease = cleaned_dataframes['disease']
billing = cleaned_dataframes['billing']
patient_insurance = cleaned_dataframes['patient_insurance']
insurance_provider = cleaned_dataframes['insurance_provider']

print("HMIS models a single hospital (no hospital_id column anywhere).")
print("hospital_id=1 / hospital_name='HMIS Hospital' used as a placeholder throughout.\n")

HMIS models a single hospital (no hospital_id column anywhere).
hospital_id=1 / hospital_name='HMIS Hospital' used as a placeholder throughout.



## Table 1 — Hospital Overview

Grain: 1 row = 1 admission. Built entirely from HMIS; `admission_source`, `patient_satisfaction_score`,
`mortality_flag` are left as NaN placeholders (no source in any of the 3 datasets provides these).
`readmission_flag` is a documented 30-day same-patient proxy, not ground truth.

In [3]:
ho = admission.copy()

# department name
ho = ho.merge(department[['department_id','department_name']], on='department_id', how='left')

# bed_type via bed -> ward -> ward_type
bed_ward = bed.merge(ward[['ward_id','ward_type']], on='ward_id', how='left')
ho = ho.merge(bed_ward[['bed_id','ward_type']].rename(columns={'ward_type':'bed_type'}), on='bed_id', how='left')

# patient_age at time of admission, patient_gender
ho = ho.merge(patient[['patient_id','date_of_birth','gender']], on='patient_id', how='left')
ho['patient_age'] = ((ho['admission_date'] - ho['date_of_birth']).dt.days // 365.25).astype(int)

# DATA-QUALITY FLAG (found by running validation, not a bug in this formula):
# some HMIS patients have admissions dated BEFORE their recorded date_of_birth
# (a flaw in the raw HMIS synthetic data), which produces an impossible
# negative age. We null the impossible value rather than keep a nonsense
# number or silently fillna(0) - per the project's own outlier-handling rule.
impossible_age = ~ho['patient_age'].between(0, 120)
print(f'[hospital_overview] rows with impossible patient_age (HMIS source-data issue): {impossible_age.sum()} / {len(ho)}')
ho.loc[impossible_age, 'patient_age'] = pd.NA

ho = ho.rename(columns={'gender':'patient_gender'})

# diagnosis
ho = ho.merge(disease[['disease_id','disease_name']], on='disease_id', how='left')
ho = ho.rename(columns={'disease_name':'diagnosis'})

# billing
ho = ho.merge(billing[['admission_id','total_amount','payment_status']], on='admission_id', how='left')
ho = ho.rename(columns={'total_amount':'total_bill_amount'})

# insurance_type: match a policy that was ACTIVE on the admission date
pi = patient_insurance.merge(insurance_provider[['insurance_provider_id','provider_type']], on='insurance_provider_id')
pi_candidates = ho[['admission_id','patient_id','admission_date']].merge(pi, on='patient_id', how='left')
active_policy = pi_candidates[
    (pi_candidates['admission_date'] >= pi_candidates['policy_start_date']) &
    (pi_candidates['admission_date'] <= pi_candidates['policy_end_date'])
].drop_duplicates(subset='admission_id')
ho = ho.merge(active_policy[['admission_id','provider_type']], on='admission_id', how='left')
ho = ho.rename(columns={'provider_type':'insurance_type'})
print(f"insurance_type matched for {ho['insurance_type'].notna().mean()*100:.1f}% of admissions (rest NaN)")

# hospital_id / hospital_name placeholders
ho['hospital_id'] = 1
ho['hospital_name'] = 'HMIS Hospital'

# readmission_flag proxy: same patient re-admitted within 30 days of prior discharge
ho = ho.sort_values(['patient_id','admission_date'])
ho['prev_discharge'] = ho.groupby('patient_id')['discharge_date'].shift(1)
gap_days = (ho['admission_date'] - ho['prev_discharge']).dt.days
ho['readmission_flag'] = ((gap_days >= 0) & (gap_days <= 30)).astype(int)
ho = ho.drop(columns=['prev_discharge'])
print(f"readmission_flag: {ho['readmission_flag'].sum()} flagged (30-day proxy, not the Readmission dataset's real flag)")

# genuinely unavailable fields
ho['admission_source'] = pd.NA
ho['patient_satisfaction_score'] = pd.NA
ho['mortality_flag'] = pd.NA
ho = ho.rename(columns={'admission_status':'discharge_status'})

hospital_overview_dataset = ho[[
    'admission_id','patient_id','hospital_id','hospital_name','department_id','department_name',
    'admission_date','discharge_date','admission_type','admission_source','bed_id','bed_type',
    'patient_age','patient_gender','diagnosis','insurance_type','total_bill_amount','payment_status',
    'discharge_status','patient_satisfaction_score','mortality_flag','readmission_flag'
]]
assert hospital_overview_dataset['admission_id'].is_unique
assert len(hospital_overview_dataset) == len(admission)
print(f"\nhospital_overview_dataset: {hospital_overview_dataset.shape} - PK unique, row count matches admission table")

[hospital_overview] rows with impossible patient_age (HMIS source-data issue): 1630 / 45000
insurance_type matched for 24.0% of admissions (rest NaN)
readmission_flag: 986 flagged (30-day proxy, not the Readmission dataset's real flag)

hospital_overview_dataset: (45000, 22) - PK unique, row count matches admission table


## Table 2 — Patient Flow

Grain: 1 row = 1 movement event. **Known limitation:** HMIS has no movement/transfer log at all - a
real patient path (ED -> Ward -> ICU -> Discharge) cannot be reconstructed. This table is limited to
2 synthetic events per admission (Admission + Discharge), same department/bed for both, since that's
the only information HMIS actually captures.

In [4]:
adm_event = admission[['admission_id','patient_id','department_id','bed_id','admission_date']].copy()
adm_event['movement_type'] = 'Admission'; adm_event['movement_sequence'] = 1
adm_event = adm_event.rename(columns={'admission_date':'movement_datetime'})

dis_event = admission[['admission_id','patient_id','department_id','bed_id','discharge_date']].copy()
dis_event['movement_type'] = 'Discharge'; dis_event['movement_sequence'] = 2
dis_event = dis_event.rename(columns={'discharge_date':'movement_datetime'})

pf = pd.concat([adm_event, dis_event], ignore_index=True)
pf = pf.sort_values(['admission_id','movement_sequence']).reset_index(drop=True)
pf['movement_id'] = range(1, len(pf) + 1)

pf = pf.merge(department[['department_id','department_name']], on='department_id', how='left')
pf = pf.rename(columns={'department_id':'current_department_id', 'department_name':'current_department_name'})
pf['from_department_id'] = pd.NA
pf['from_department_name'] = pd.NA

pf['hospital_id'] = 1
pf['movement_date'] = pf['movement_datetime'].dt.date

# duration_in_department_hours: only meaningful for the Admission row = full LOS in hours
los_hours = (admission['discharge_date'] - admission['admission_date']).dt.total_seconds() / 3600
los_map = dict(zip(admission['admission_id'], los_hours))
pf['duration_in_department_hours'] = pf.apply(
    lambda r: los_map[r['admission_id']] if r['movement_type'] == 'Admission' else 0, axis=1
)

pf['year'] = pf['movement_datetime'].dt.year
pf['month'] = pf['movement_datetime'].dt.month
pf['day_of_week'] = pf['movement_datetime'].dt.day_name()
pf['hour_of_day'] = pf['movement_datetime'].dt.hour  # note: source dates have no real time component, will be 0
pf['shift'] = pd.cut(pf['hour_of_day'], bins=[-1,7,15,23], labels=['Night','Morning','Evening'])
pf['is_peak_hour'] = pf['hour_of_day'].between(9, 17)

patient_flow_dataset = pf[[
    'movement_id','admission_id','patient_id','hospital_id','movement_sequence','movement_type',
    'from_department_id','current_department_id','from_department_name','current_department_name',
    'bed_id','movement_datetime','movement_date','duration_in_department_hours',
    'year','month','day_of_week','hour_of_day','shift','is_peak_hour'
]]
assert patient_flow_dataset['movement_id'].is_unique
assert len(patient_flow_dataset) == len(admission) * 2
print(f"patient_flow_dataset: {patient_flow_dataset.shape} - PK unique, 2 events per admission confirmed")

patient_flow_dataset: (90000, 20) - PK unique, 2 events per admission confirmed


## Table 3 — Department Analytics

Grain: 1 row = 1 department + 1 day, scoped to the **6 clinical departments** - the 5 Admin/Diagnostic
departments (Radiology, Pathology, Pharmacy, Billing, HR) have no wards/beds and never receive
admissions, so a bed-based daily table for them would be meaningless zeros.

`occupied_beds_count` is computed with an event-delta + cumulative-sum trick: +1 on admission date,
-1 the day after discharge, summed per department per day.

In [5]:
CLINICAL_DEPT_IDS = ward['department_id'].unique()  # only depts that actually have wards/beds

events = pd.concat([
    admission[['department_id','admission_date']].rename(columns={'admission_date':'date'}).assign(delta=1),
    admission.assign(day_after_discharge=admission['discharge_date'] + pd.Timedelta(days=1))
             [['department_id','day_after_discharge']].rename(columns={'day_after_discharge':'date'}).assign(delta=-1)
])
daily_delta = events.groupby(['department_id','date'])['delta'].sum().reset_index()

full_date_range = pd.date_range(admission['admission_date'].min(), admission['discharge_date'].max(), freq='D')
idx = pd.MultiIndex.from_product([CLINICAL_DEPT_IDS, full_date_range], names=['department_id','date'])
da = daily_delta.set_index(['department_id','date']).reindex(idx, fill_value=0).reset_index()
da = da.sort_values(['department_id','date'])
da['occupied_beds_count'] = da.groupby('department_id')['delta'].cumsum()
da = da.drop(columns=['delta'])

total_beds_by_dept = ward.groupby('department_id')['total_beds'].sum().reset_index()
da = da.merge(total_beds_by_dept, on='department_id', how='left')
da = da.merge(department[['department_id','department_name','department_type']], on='department_id', how='left')
da['bed_occupancy_rate_pct'] = (da['occupied_beds_count'] / da['total_beds'] * 100).round(2)

admitted_counts = admission.groupby(['department_id','admission_date']).size().reset_index(name='patients_admitted_count')
admitted_counts = admitted_counts.rename(columns={'admission_date':'date'})
discharged_counts = admission.groupby(['department_id','discharge_date']).size().reset_index(name='patients_discharged_count')
discharged_counts = discharged_counts.rename(columns={'discharge_date':'date'})
da = da.merge(admitted_counts, on=['department_id','date'], how='left')
da = da.merge(discharged_counts, on=['department_id','date'], how='left')
da[['patients_admitted_count','patients_discharged_count']] = da[['patients_admitted_count','patients_discharged_count']].fillna(0).astype(int)

adm_los = admission.copy()
adm_los['los_days'] = (adm_los['discharge_date'] - adm_los['admission_date']).dt.days
los_by_day = adm_los.groupby(['department_id','discharge_date'])['los_days'].mean().reset_index()
los_by_day = los_by_day.rename(columns={'discharge_date':'date','los_days':'avg_length_of_stay_days'})
da = da.merge(los_by_day, on=['department_id','date'], how='left')

readmit_agg = ho.groupby(['department_id','discharge_date'])['readmission_flag'].agg(['sum','count']).reset_index()
readmit_agg = readmit_agg.rename(columns={'discharge_date':'date','sum':'readmission_count'})
readmit_agg['readmission_rate_pct'] = (readmit_agg['readmission_count'] / readmit_agg['count'] * 100).round(2)
da = da.merge(readmit_agg[['department_id','date','readmission_count','readmission_rate_pct']], on=['department_id','date'], how='left')

da['hospital_id'] = 1
da['hospital_name'] = 'HMIS Hospital'

# genuinely missing fields - left as NaN placeholders for when other datasets are merged
for col in ['mortality_count','mortality_rate_pct','avg_treatment_time_hours','transfer_events_count',
            'nurses_on_duty','doctors_on_duty','staff_to_patient_ratio','equipment_downtime_hours',
            'avg_satisfaction_score','department_efficiency_score']:
    da[col] = pd.NA

department_analytics_dataset = da[[
    'date','hospital_id','hospital_name','department_id','department_name','department_type',
    'total_beds','occupied_beds_count','bed_occupancy_rate_pct','patients_admitted_count',
    'patients_discharged_count','readmission_count','readmission_rate_pct','mortality_count',
    'mortality_rate_pct','avg_length_of_stay_days','avg_treatment_time_hours','transfer_events_count',
    'nurses_on_duty','doctors_on_duty','staff_to_patient_ratio','equipment_downtime_hours',
    'avg_satisfaction_score','department_efficiency_score'
]]
assert department_analytics_dataset.duplicated(subset=['department_id','date']).sum() == 0
assert (department_analytics_dataset['occupied_beds_count'] >= 0).all()
assert (department_analytics_dataset['occupied_beds_count'] <= department_analytics_dataset['total_beds']).all()
print(f"department_analytics_dataset: {department_analytics_dataset.shape} - grain unique, occupancy within [0, total_beds]")

department_analytics_dataset: (13224, 24) - grain unique, occupancy within [0, total_beds]


## Table 4 — Resource Utilization

Grain: 1 row = 1 department + 1 day + 1 resource type. **Only `Bed` rows exist** - Equipment (no
equipment table anywhere in HMIS) and dated Clinical Staff (`staff_assignment` has no date column,
only ward+shift, so a per-day count can't be produced from HMIS alone) are structural gaps.

In [6]:
ru = da[['department_id','department_name','date','total_beds','occupied_beds_count','bed_occupancy_rate_pct']].copy()
ru = ru.rename(columns={'total_beds':'total_units_available','occupied_beds_count':'units_in_use',
                         'bed_occupancy_rate_pct':'utilization_rate_pct'})
ru['resource_type'] = 'Bed'
ru['resource_category'] = pd.NA
ru['hospital_id'] = 1
ru['hospital_name'] = 'HMIS Hospital'
ru['units_under_maintenance'] = pd.NA
ru['shortage_flag'] = ru['utilization_rate_pct'] > 90
ru['capacity_hours'] = ru['total_units_available'] * 24
ru['utilized_hours'] = ru['units_in_use'] * 24
ru['idle_hours'] = ru['capacity_hours'] - ru['utilized_hours']
ru['downtime_hours'] = pd.NA
ru['resource_utilization_id'] = range(1, len(ru) + 1)

resource_utilization_dataset = ru[[
    'resource_utilization_id','date','hospital_id','hospital_name','department_id','department_name',
    'resource_type','resource_category','total_units_available','units_in_use','units_under_maintenance',
    'utilization_rate_pct','shortage_flag','capacity_hours','utilized_hours','idle_hours','downtime_hours'
]]
assert resource_utilization_dataset['resource_utilization_id'].is_unique
print(f"resource_utilization_dataset: {resource_utilization_dataset.shape} - PK unique")

before_rows = {'department_analytics': len(department_analytics_dataset), 'resource_utilization': len(resource_utilization_dataset)}

resource_utilization_dataset: (13224, 17) - PK unique


## Bridge 1 — Beds Management -> Department Analytics + Resource Utilization

No shared key with HMIS. Bridged via two dimensions instead of a row-level join:
- **Department name bridge**: Beds Management's 4 services map to 4 of HMIS's 6 clinical departments
  (`Pediatrics`, `Orthopedics` have no counterpart and stay NaN).
- **Time bridge**: `services_weekly`/`staff_schedule` only give week number + month, no year or day.
  Verified against the dataset's own `month` column (0 mismatches) and checked for zero overlapping
  date ranges before trusting it.

Adds `avg_satisfaction_score`, `nurses_on_duty`, `doctors_on_duty` to Department Analytics, and
benchmark bed/demand columns to Resource Utilization - always as a LEFT JOIN, HMIS stays authoritative.

In [7]:
beds_services_weekly = pd.read_csv(BEDS_PROCESSED_DIR / "beds_services_weekly.csv")
beds_staff_schedule  = pd.read_csv(BEDS_PROCESSED_DIR / "beds_staff_schedule.csv")

SERVICE_TO_DEPARTMENT_NAME = {
    'emergency': 'Emergency', 'surgery': 'Surgery', 'ICU': 'ICU',
    'general_medicine': 'Internal Medicine',   # ASSUMPTION - only reasonable match
}
department_lookup = department[['department_id', 'department_name']]

def map_service_to_department_id(service_value):
    dept_name = SERVICE_TO_DEPARTMENT_NAME.get(service_value)
    if dept_name is None:
        return pd.NA
    match = department_lookup.loc[department_lookup['department_name'] == dept_name, 'department_id']
    return match.iloc[0] if len(match) else pd.NA

print("service -> department_name mapping:")
for k, v in SERVICE_TO_DEPARTMENT_NAME.items():
    print(f"  {k} -> {v}")
print("  Pediatrics / Orthopedics -> no match in Beds Management (stay NaN)")

service -> department_name mapping:
  emergency -> Emergency
  surgery -> Surgery
  ICU -> ICU
  general_medicine -> Internal Medicine
  Pediatrics / Orthopedics -> no match in Beds Management (stay NaN)


In [8]:
# --- week -> calendar date bridge, verified before use ---
ASSUMED_YEAR = 2025
week_month_lookup = beds_services_weekly[['week', 'month']].drop_duplicates()

weeks_with_multiple_months = week_month_lookup.groupby('week')['month'].nunique()
assert (weeks_with_multiple_months <= 1).all(), "STOP: a week maps to multiple months"

# Give each week a VARIABLE length so weeks in a month partition it exactly
# (no overlap, no gap) - a fixed 7-day-per-week assumption caused duplicate
# (department_id, date) rows in an earlier version of this bridge.
week_date_info = {}
for month, group in week_month_lookup.groupby('month'):
    weeks_in_this_month = sorted(group['week'].tolist())
    n_weeks = len(weeks_in_this_month)
    days_in_this_month = calendar.monthrange(ASSUMED_YEAR, month)[1]
    base_len = days_in_this_month // n_weeks
    remainder = days_in_this_month % n_weeks
    cursor = pd.Timestamp(ASSUMED_YEAR, month, 1)
    for i, week_num in enumerate(weeks_in_this_month):
        this_week_len = base_len + (1 if i < remainder else 0)
        week_date_info[week_num] = (cursor, this_week_len)
        cursor = cursor + pd.Timedelta(days=this_week_len)

def week_to_start_date(w): return week_date_info[w][0]
def week_to_length(w): return week_date_info[w][1]

beds_services_weekly['week_start_date'] = beds_services_weekly['week'].apply(week_to_start_date)
beds_services_weekly['week_length'] = beds_services_weekly['week'].apply(week_to_length)
beds_staff_schedule['week_start_date'] = beds_staff_schedule['week'].apply(week_to_start_date)
beds_staff_schedule['week_length'] = beds_staff_schedule['week'].apply(week_to_length)

derived_month = beds_services_weekly['week_start_date'].dt.month
mismatch_count = (derived_month != beds_services_weekly['month']).sum()
assert mismatch_count == 0, "STOP: month alignment broken"

all_ranges = []
for wk, (start, length) in week_date_info.items():
    for offset in range(length):
        all_ranges.append(start + pd.Timedelta(days=offset))
overlap_count = pd.Series(all_ranges).duplicated().sum()
assert overlap_count == 0, "STOP: week ranges overlap"
print(f"Date bridge verified: 0 month mismatches, 0 overlaps, {len(all_ranges)} days covered (expect 365)")

Date bridge verified: 0 month mismatches, 0 overlaps, 365 days covered (expect 365)


In [9]:
def expand_week_to_days(df, value_cols):
    """Expand each weekly row into one row per day, using each week's
    REAL length (not a hardcoded 7) so no two weeks' date ranges overlap."""
    rows = []
    for _, row in df.iterrows():
        for offset in range(int(row['week_length'])):
            new_row = row[value_cols].to_dict()
            new_row['date'] = row['week_start_date'] + pd.Timedelta(days=offset)
            rows.append(new_row)
    return pd.DataFrame(rows)

# --- services_weekly -> daily ---
beds_services_weekly['department_id'] = beds_services_weekly['service'].apply(map_service_to_department_id)
services_daily = expand_week_to_days(
    beds_services_weekly,
    value_cols=['department_id','available_beds','patients_refused','patient_satisfaction','week_length']
)
services_daily = services_daily.dropna(subset=['department_id'])
services_daily['department_id'] = services_daily['department_id'].astype(int)
services_daily = services_daily.rename(columns={
    'patient_satisfaction': 'new_avg_satisfaction_score',
    'available_beds': 'new_external_benchmark_available_beds',
    'patients_refused': 'new_external_benchmark_patients_refused',
})
assert services_daily.duplicated(subset=['department_id','date']).sum() == 0, "STOP: services_daily has dup keys"
print(f"services_daily: {services_daily.shape}")

# --- staff_schedule -> weekly counts per department+role -> daily ---
beds_staff_schedule['department_id'] = beds_staff_schedule['service'].apply(map_service_to_department_id)
weekly_staff_counts = (
    beds_staff_schedule[beds_staff_schedule['present'] == 1]
    .groupby(['week','week_start_date','week_length','department_id','role']).size().reset_index(name='count')
)
staff_pivot = weekly_staff_counts.pivot_table(
    index=['week','week_start_date','week_length','department_id'], columns='role', values='count', fill_value=0
).reset_index()
rename_map = {}
if 'doctor' in staff_pivot.columns: rename_map['doctor'] = 'new_doctors_on_duty'
if 'nurse' in staff_pivot.columns: rename_map['nurse'] = 'new_nurses_on_duty'
for role_col in staff_pivot.columns:
    if role_col not in ('week','week_start_date','week_length','department_id') and role_col not in rename_map:
        rename_map[role_col] = f'new_{role_col}_on_duty'
staff_pivot = staff_pivot.rename(columns=rename_map)
staff_value_cols = ['department_id','week_length'] + [c for c in staff_pivot.columns if c.startswith('new_')]
staff_daily = expand_week_to_days(staff_pivot, value_cols=staff_value_cols)
staff_daily = staff_daily.dropna(subset=['department_id'])
staff_daily['department_id'] = staff_daily['department_id'].astype(int)
assert staff_daily.duplicated(subset=['department_id','date']).sum() == 0, "STOP: staff_daily has dup keys"
print(f"staff_daily: {staff_daily.shape}")

services_daily: (1460, 6)
staff_daily: (984, 6)


staff_daily: (984, 6)


In [10]:
# --- merge into department_analytics_dataset ---
da2 = department_analytics_dataset.copy()
da2 = da2.merge(services_daily, on=['department_id','date'], how='left')
da2 = da2.merge(staff_daily, on=['department_id','date'], how='left')

da2['avg_satisfaction_score'] = da2['avg_satisfaction_score'].combine_first(da2['new_avg_satisfaction_score'])
if 'new_doctors_on_duty' in da2.columns:
    da2['doctors_on_duty'] = da2['doctors_on_duty'].combine_first(da2['new_doctors_on_duty'])
if 'new_nurses_on_duty' in da2.columns:
    da2['nurses_on_duty'] = da2['nurses_on_duty'].combine_first(da2['new_nurses_on_duty'])

da2['external_benchmark_available_beds'] = da2['new_external_benchmark_available_beds']
da2['external_benchmark_patients_refused'] = da2['new_external_benchmark_patients_refused']

mask = da2['nurses_on_duty'].notna() & da2['doctors_on_duty'].notna() & (da2['patients_admitted_count'] > 0)
da2.loc[mask, 'staff_to_patient_ratio'] = (
    (da2.loc[mask,'nurses_on_duty'] + da2.loc[mask,'doctors_on_duty']) / da2.loc[mask,'patients_admitted_count']
)

# Re-select the EXACT target schema instead of just dropping 'new_*' columns.
# expand_week_to_days() also carried a 'week_length' helper column through
# from BOTH services_daily and staff_daily, which the merge auto-suffixed to
# week_length_x/week_length_y and leaked into an earlier saved CSV. Explicit
# reselect guarantees no stray column can ever leak into the final table.
department_analytics_final_columns = [
    'date','hospital_id','hospital_name','department_id','department_name','department_type',
    'total_beds','occupied_beds_count','bed_occupancy_rate_pct','patients_admitted_count',
    'patients_discharged_count','readmission_count','readmission_rate_pct','mortality_count',
    'mortality_rate_pct','avg_length_of_stay_days','avg_treatment_time_hours','transfer_events_count',
    'nurses_on_duty','doctors_on_duty','staff_to_patient_ratio','equipment_downtime_hours',
    'avg_satisfaction_score','department_efficiency_score',
    'external_benchmark_available_beds','external_benchmark_patients_refused',
]
da2 = da2[department_analytics_final_columns]

assert len(da2) == before_rows['department_analytics'], "ROW COUNT CHANGED - stop and investigate"
print(f"department_analytics_dataset row count unchanged: {len(da2)} rows")
print(f"avg_satisfaction_score filled for {da2['avg_satisfaction_score'].notna().sum()}/{len(da2)} rows")
department_analytics_dataset = da2

department_analytics_dataset row count unchanged: 13224 rows
avg_satisfaction_score filled for 1460/13224 rows


C:\Users\sr189\AppData\Local\Temp\ipykernel_4004\869945715.py:6: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  da2['avg_satisfaction_score'] = da2['avg_satisfaction_score'].combine_first(da2['new_avg_satisfaction_score'])
C:\Users\sr189\AppData\Local\Temp\ipykernel_4004\869945715.py:8: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  da2['doctors_on_duty'] = da2['doctors_on_duty'].combine_first(da2['new_doctors_on_duty'])
C:\Users\sr189\AppData\Local\Temp\ipykernel_4004\869945715.py:10: FutureWarning: The behavior of array concatenation with empty entries i

In [11]:
# --- merge into resource_utilization_dataset ---
ru2 = resource_utilization_dataset.copy()
bench = services_daily[['department_id','date','new_external_benchmark_available_beds','new_external_benchmark_patients_refused']]
ru2 = ru2.merge(bench, on=['department_id','date'], how='left')
ru2 = ru2.rename(columns={
    'new_external_benchmark_available_beds':'external_benchmark_available_beds',
    'new_external_benchmark_patients_refused':'external_benchmark_patients_refused',
})
assert len(ru2) == before_rows['resource_utilization'], "ROW COUNT CHANGED - stop and investigate"
print(f"resource_utilization_dataset row count unchanged: {len(ru2)} rows")
resource_utilization_dataset = ru2

print("\nhospital_overview_dataset / patient_flow_dataset: unchanged, not merged")
print("(admission-grain, no row-level bridge exists to Beds Management's unrelated patients)")

resource_utilization_dataset row count unchanged: 13224 rows

hospital_overview_dataset / patient_flow_dataset: unchanged, not merged
(admission-grain, no row-level bridge exists to Beds Management's unrelated patients)


## Bridge 2 — Readmission dataset -> Hospital Overview (disease-level benchmark)

No shared key with HMIS. Bridged via disease name only (7 of 19 HMIS diseases match exactly after
stripping parenthetical abbreviations like "(COPD)"; the rest use the dataset-wide average, clearly
labeled via `benchmark_match_type` so nothing is silently treated as disease-specific when it isn't).

Adds `benchmark_mortality_rate`, `benchmark_readmission_rate`, `benchmark_satisfaction_score` to
Hospital Overview ONLY - Department Analytics/Resource Utilization get nothing from this dataset,
since `department_referral`/`hospital_ward` use a completely different taxonomy than HMIS's own
`department_name`/`ward_type` and no honest mapping exists.

In [12]:
readmission_df = pd.read_csv(READMISSION_PROCESSED_DIR / "readmission_dataset.csv")
disease2 = disease.copy()

def normalize_disease_name(name):
    """Strips a trailing parenthetical abbreviation only, e.g.
    'Chronic Obstructive Pulmonary Disease (COPD)' -> 'Chronic
    Obstructive Pulmonary Disease'. Never touches the original column."""
    return re.sub(r'\s*\([^)]*\)\s*$', '', str(name)).strip()

readmission_df['disease_normalized'] = readmission_df['patient_disease'].apply(normalize_disease_name)
disease2['disease_normalized'] = disease2['disease_name'].apply(normalize_disease_name)
matched_diseases = set(readmission_df['disease_normalized'].unique()) & set(disease2['disease_normalized'].unique())
print(f"Diseases matched after normalization: {len(matched_diseases)} / {disease2['disease_name'].nunique()} HMIS diseases")
print(matched_diseases)

Diseases matched after normalization: 7 / 20 HMIS diseases
{'Chronic Kidney Disease', 'Hypertension', 'Urinary Tract Infection', 'COVID-19', 'Pneumonia', 'Chronic Obstructive Pulmonary Disease', 'Stroke'}


In [13]:
per_disease_stats = readmission_df.groupby('disease_normalized').agg(
    benchmark_mortality_rate=('discharge_status', lambda s: (s == 'Deceased').mean()),
    benchmark_readmission_rate=('readmission', 'mean'),
    benchmark_satisfaction_score=('patient_sat_score', lambda s: s.mean() / 16),  # /16: documented assumption, 1600-scale looks SAT-style
    benchmark_sample_size=('patient_disease', 'count')
).reset_index()

overall_stats = {
    'benchmark_mortality_rate': (readmission_df['discharge_status'] == 'Deceased').mean(),
    'benchmark_readmission_rate': readmission_df['readmission'].mean(),
    'benchmark_satisfaction_score': readmission_df['patient_sat_score'].mean() / 16,
    'benchmark_sample_size': len(readmission_df),
}

benchmark_rows = []
for _, row in disease2.iterrows():
    hmis_disease_name = row['disease_name']
    normalized = row['disease_normalized']
    if normalized in matched_diseases:
        stats = per_disease_stats.loc[per_disease_stats['disease_normalized'] == normalized].iloc[0]
        benchmark_rows.append({'disease_name': hmis_disease_name,
                                'benchmark_mortality_rate': stats['benchmark_mortality_rate'],
                                'benchmark_readmission_rate': stats['benchmark_readmission_rate'],
                                'benchmark_satisfaction_score': stats['benchmark_satisfaction_score'],
                                'benchmark_sample_size': stats['benchmark_sample_size'],
                                'benchmark_match_type': 'disease-specific'})
    else:
        benchmark_rows.append({'disease_name': hmis_disease_name, **overall_stats,
                                'benchmark_match_type': 'dataset-average fallback'})

disease_outcome_benchmarks = pd.DataFrame(benchmark_rows)
print(f"disease_outcome_benchmarks: {len(disease_outcome_benchmarks)} rows "
      f"({(disease_outcome_benchmarks['benchmark_match_type']=='disease-specific').sum()} disease-specific, "
      f"{(disease_outcome_benchmarks['benchmark_match_type']=='dataset-average fallback').sum()} fallback)")
disease_outcome_benchmarks.to_csv(PROCESSED_DIR / "disease_outcome_benchmarks.csv", index=False)

disease_outcome_benchmarks: 20 rows (7 disease-specific, 13 fallback)


In [14]:
before_ho_rows = len(hospital_overview_dataset)
ho2 = hospital_overview_dataset.merge(disease_outcome_benchmarks, left_on='diagnosis', right_on='disease_name', how='left').drop(columns=['disease_name'])
assert len(ho2) == before_ho_rows, "ROW COUNT CHANGED - stop and investigate"
hospital_overview_dataset = ho2
print(f"hospital_overview_dataset row count unchanged: {len(ho2)} rows")
print(f"benchmark columns filled for {ho2['benchmark_mortality_rate'].notna().sum()}/{len(ho2)} rows")

hospital_overview_dataset row count unchanged: 45000 rows
benchmark columns filled for 45000/45000 rows


## Save the Four Final Tables

In [15]:
hospital_overview_dataset.to_csv(PROCESSED_DIR / "hospital_overview_dataset.csv", index=False)
patient_flow_dataset.to_csv(PROCESSED_DIR / "patient_flow_dataset.csv", index=False)
department_analytics_dataset.to_csv(PROCESSED_DIR / "department_analytics_dataset.csv", index=False)
resource_utilization_dataset.to_csv(PROCESSED_DIR / "resource_utilization_dataset.csv", index=False)
print("Saved all 4 final tables to", PROCESSED_DIR)

Saved all 4 final tables to C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed
